# Rust — Le bilan « mathocalypse » — quand l'IA publie 372 théorèmes en un soir

**Série Complexity — accrétion 07c** · Public : **Recherche** · Hommage fondateur : **Scott Aaronson** (« The Mathocalypse », Shtetl-Optimized, 7/10/2026, mise à jour 12/10/2026)

**Navigation** : [<< Complexity-07b — Le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) | [Index de la série](README.md) | [Version Python](Complexity-07c-LeBilanMathocalypse-Python.ipynb) | [Version C#](Complexity-07c-LeBilanMathocalypse-CSharp.ipynb)

Le 6 octobre 2026, OpenAI publie d'un coup **372 résultats de recherche** obtenus par ses agents, sur recommandation de son groupe consultatif AGMAI (Timothy Gowers, Edward Witten et autres mathématiciens). Ce carnet décrypte cette vague pour l'étudiant : ce qui a été prouvé, ce que ça change, ce que ça ne change pas — et ce que « comprendre une preuve » veut dire quand plus personne ne la comprend.

**Distinction de périmètre, écrite ici comme exigé** : ce carnet fait **comprendre** les résultats (décryptage, Zoo, mesures numériques) ; la **distillation formelle** relève de l'arbitrage formel du dépôt, dont le pli 1 (09/10/2026) classe ces résultats précis en **non-distillation écrite** — motif : aucun lac du dépôt ne porte la théorie correspondante — et confie la matière **en veille à cette série-ci** : ce carnet est ce premier dépôt, il renvoie, il ne refait pas. Conventions de la série : chaque fait est **Mesuré** (sortie de ce carnet) ou **Cité** (source nominative). Source primaire archivée au gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\AI and Mathematics - the October 2026 Release and its Reception\2026 - Aaronson - blog post 10169 sur la release openai-math (scottaaronson.blog, OA).html`.

**Jumeau Rust.** Les calculs et les bancs 3SUM sont natifs, écrits avec la seule bibliothèque standard Rust (aucune crate externe, aucun `:dep`). Le générateur d'aléa rejoue le flux MT19937 de CPython — même graine, même suite de mots, rejets compris — si bien que les tirages de ce carnet sont ceux de la source ; les chronos, eux, sont propres à la machine. **Ce que Rust apporte** est mesuré en section 5 : le même 3SUM sur la même liste, chronométré en Python et en Rust sur la même machine. Le Python versionné est la source primaire consultée ; les grands résultats et citations sont rapportés de son billet cité, sans nouvelle preuve vérifiée ici. [Source Python](Complexity-07c-LeBilanMathocalypse-Python.ipynb). Le jumeau du même fil en C# : [version C#](Complexity-07c-LeBilanMathocalypse-CSharp.ipynb).


## 1. La mécanique de la vague — ce qui a tourné, combien de temps

Le récit factuel du billet : des agents GPT-Pro ont reçu environ 4 000 problèmes ouverts de longue date ; environ 10 % ont abouti, donnant les 372 résultats publiés. Le coût moyen : **environ 3 heures de calcul de niveau GPT-Pro par problème résolu**. Une correction honnête vit dans le fil : une première version du billet annonçait 8 000 problèmes et ~5 % de succès, corrigée par les commentateurs (la source officielle, `github.com/openai/math`, dit ~4 000 problèmes posés) — même les témoins directs se trompent dans l'urgence, et le corriger en public est la moitié de l'épistémologie de ce carnet.


In [2]:
// Les trois hypotheses du billet, reprises telles quelles de la source.
const PBM_POSES: i64 = 4000;        // ~problemes poses (Cite : github.com/openai/math, via le billet)
const RESOLUS: i64 = 372;            // resultats publies (Cite)
const H_PAR_RESOLU: i64 = 3;         // heures de GPT-Pro par probleme RESOLU (Cite : "per problem solved")

// Groupement des milliers, comme f"{x:,}" en Python : virgules tous les trois chiffres
// depuis la droite. Rust n'a pas d'equivalent natif.
fn py_group(s: &str) -> String {
    let (signe, chiffres) = match s.strip_prefix('-') {
        Some(reste) => ("-", reste),
        None => ("", s),
    };
    let n = chiffres.len();
    let mut sortie = String::new();
    for (i, c) in chiffres.chars().enumerate() {
        if i > 0 && (n - i) % 3 == 0 {
            sortie.push(',');
        }
        sortie.push(c);
    }
    format!("{}{}", signe, sortie)
}

let taux = RESOLUS as f64 / PBM_POSES as f64;
println!("Taux de resolution : {:.3} soit ~{:.1}% (Mesure)", taux, taux * 100.0);
let heures = RESOLUS * H_PAR_RESOLU;
println!("Compute des problemes resolus : {} h GPT-Pro (Mesure)", py_group(&heures.to_string()));
println!("Ordre de grandeur : {:.0} jours d'un seul agent non stop (Mesure)", heures as f64 / 24.0);
println!("Fenetre : une premiere version disait 8000 problemes / ~5% - corrigee en public (Cite)");


Taux de resolution : 0.093 soit ~9.3% (Mesure)


Compute des problemes resolus : 1,116 h GPT-Pro (Mesure)


Ordre de grandeur : 46 jours d'un seul agent non stop (Mesure)


Fenetre : une premiere version disait 8000 problemes / ~5% - corrigee en public (Cite)


## 2. Unique Games — la conjecture qui est tombée, et la mathématicienne « cooked »

La pièce maîtresse de la vague : une preuve de la **conjecture Unique Games** de Subhash Khot (UGC, 2002) — l'un des grands problèmes d'optimisation, qui gouverne la dureté d'approximation de Max Cut et de toutes les CSP. Le billet raconte que le fils de la complexity theorist **Dana Moshkovitz** lui a lancé : *« mommy, I heard you got cooked! »* — elle travaillait vers cette conjecture depuis des années ; le billet publie d'ailleurs sa réaction à chaud (commentaire #5 : elle doute que le « teleport to the peak » suffise à comprendre la montagne). La preuve, elle, « invente un code complètement nouveau et bizarre avec un test de bruit », une récursion folle — et les preuves d'inapproximabilité des applications (Max Cut, CSP) contournent désormais l'UGC elles-mêmes (Cité — billet, §résultats).


## 3. Derandomization et quantique — L=BPL, parité ∉ QAC$^0$, area law

Trois questions de trente ans et plus, tombées dans la même fournée : **L=BPL** (le logspace probabiliste égale le logspace déterministe — l'une des grandes conjectures de dérandomization) ; **la parité n'est pas dans QAC$^0$** (question ouverte de complexité quantique depuis 1999, sur laquelle « many of my colleagues had been closing in on ») ; et la **loi d'aire pour Hamiltoniens gapés 2D**, l'un des problèmes ouverts majeurs de la complexité hamiltonienne. Une seule des trois a sa case au [Complexity Zoo du 03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) : **L** — la carte du 03c s'arrête à $L \subseteq NL \subseteq P \subseteq NP \subseteq PSPACE$ (avec EXPTIME et coNP) et ne porte **ni BPL ni QAC$^0$**. Le Zoo reste la carte de lecture de la hiérarchie que la vague traverse ; BPL et QAC$^0$ restent à lire ailleurs.


In [3]:
// Mini-zoo des questions touchees par la vague : (nom, avant, apres, role).
// Vec et non BTreeMap : les cles de la source ne sont pas triees
// (le premier temoin est "UGC (Khot 2002)", un BTreeMap mettrait "L = BPL" en tete)
// et l'ordre d'insertion fait partie du temoin de conformite.
let zoo: Vec<(&str, (&str, &str, &str))> = vec![
    ("UGC (Khot 2002)",        ("conjecture ouverte",   "prouvee par la vague", "implique durete d'approximation Max Cut/CSP")),
    ("L = BPL",                 ("conjecture ouverte",   "prouve",               "derandomization du logspace")),
    ("parite hors QAC^0",       ("ouverte depuis 1999",  "prouve",               "bornes inferieures circuits quantiques")),
    ("loi d'aire 2D gappee",    ("probleme majeur ouvert", "prouve",             "complexite hamiltonienne")),
    ("P vs NP",                 ("ouvert",               "toujours ouvert",      "absent de la liste - cf. section 6")),
];
println!("{} entrees du mini-zoo (Mesure) :", zoo.len());
for &(nom, (avant, apres, _role)) in &zoo {
    // Le role (3e champ) est porte mais jamais imprime, comme dans la source.
    println!("  {:28} {:22} -> {}", nom, avant, apres);
}
// Compte CALCULE, jamais ecrit en dur : une question est "bougee" quand son etat
// a change et qu'elle n'est pas restee "toujours ouvert".
let cassees = zoo.iter().filter(|e| e.1.1 != e.1.0 && e.1.1 != "toujours ouvert").count();
println!("Conjectures/questions bougees : {}/{} (Mesure)", cassees, zoo.len());


5 entrees du mini-zoo (Mesure) :


  UGC (Khot 2002)              conjecture ouverte     -> prouvee par la vague


  L = BPL                      conjecture ouverte     -> prouve


  parite hors QAC^0            ouverte depuis 1999    -> prouve


  loi d'aire 2D gappee         probleme majeur ouvert -> prouve


  P vs NP                      ouvert                 -> toujours ouvert


Conjectures/questions bougees : 4/5 (Mesure)


## 4. Les barrières algorithmiques — un exposant rationnel, et treize 9

Deux résultats cassent des **barrières d'exposant** vieilles de décennies : le **produit matriciel en $n^{9/4+o(1)}$** — un exposant *rationnel* pour une fois, obtenu par une approche « complètement différente » de la lignée $O(n^{2.373})$ ; et la **multiplication d'entiers en $O(n \log^{0.9999999999999} n)$** — en dessous de la barrière $n \log n$ qui tenait depuis les années 1960 (Knuth-Schönhage-Strassen). Deux subtilités que le fil du billet rend vivantes : le résultat matriciel **n'est pas constructif** (commentaire de Bram Cohen : une existence de type Ramsey, pas un algorithme qu'on peut exécuter) ; et l'exposant $0.9999999999999$ — **treize neuf** — dit une victoire asymptotique totalement invisible en pratique. On mesure les deux :


In [4]:
// Aucun `use` : la prelude Rust suffit, aucune crate externe dans ce carnet.

// Groupement des milliers, comme f"{x:,}" en Python : ARRONDIR D'ABORD (en {:.0}),
// puis inserer les virgules -- une troncature vers l'entier donnerait "12" la ou
// Python affiche "13".
fn py_group(s: &str) -> String {
    let (signe, chiffres) = match s.strip_prefix('-') {
        Some(reste) => ("-", reste),
        None => ("", s),
    };
    let n = chiffres.len();
    let mut sortie = String::new();
    for (i, c) in chiffres.chars().enumerate() {
        if i > 0 && (n - i) % 3 == 0 {
            sortie.push(',');
        }
        sortie.push(c);
    }
    format!("{}{}", signe, sortie)
}

// Exposant scientifique, comme f"{x:.0e}" en Python : "1e+06" (signe, au moins
// deux chiffres) -- le format Rust "{:.0e}" rendrait "1e6".
fn py_exp0(x: f64) -> String {
    let s = format!("{:.0e}", x);
    let mut parties = s.splitn(2, 'e');
    let mantisse = parties.next().unwrap_or("0").to_string();
    let expo = parties.next().unwrap_or("0");
    let signe = if expo.starts_with('-') { "-" } else { "+" };
    let expo_abs: u32 = expo
        .trim_start_matches('+')
        .trim_start_matches('-')
        .parse()
        .unwrap_or(0);
    format!("{}e{}{:02}", mantisse, signe, expo_abs)
}

// L'ecart d'exposant, CALCULE (jamais le litteral 0.123).
let d = 2.373f64 - 2.25f64;

// (a) Matmul : le facteur asymptotique n^2.373 -> n^2.25
for n in [1e6f64, 1e9, 1e12] {
    let facteur = n.powf(d);
    println!(
        "matmul n={} : facteur asymptotique gagne = n^0.123 = {}x (Mesure)",
        py_exp0(n),
        py_group(&format!("{:.0}", facteur))
    );
}

// ... mais avec un surcout de constante 100x (plausible pour un schema recursif neuf) :
let n_bascule = 10f64.powf((100f64).log10() / d);
println!(
    "si le nouvel algo coute 100x plus par operation, il ne gagne qu'a partir de n ~ 10^{:.1} (Mesure)",
    n_bascule.log10()
);

// (b) Entiers : log^0.9999999999999 vs log - l'ecart sur des tailles reelles
let expo = 0.9999999999999f64;   // treize neuf (Cite, billet)
for bits in [64.0f64, 1e6] {
    let log2n = bits;             // n = 2**bits, donc log2(n) = bits
    let ancien = log2n.powf(1.0);
    let nouveau = log2n.powf(expo);
    println!(
        "entiers n=2^{}: ancien ~{:.1}, nouveau ~{:.15}, gain {:.15}x (Mesure)",
        bits, ancien, nouveau, ancien / nouveau
    );
}
println!("-> la victoire asymptotique est exacte mais invisible a toute taille realiste (Mesure)");

matmul n=1e+06 : facteur asymptotique gagne = n^0.123 = 5x (Mesure)


matmul n=1e+09 : facteur asymptotique gagne = n^0.123 = 13x (Mesure)


matmul n=1e+12 : facteur asymptotique gagne = n^0.123 = 30x (Mesure)


si le nouvel algo coute 100x plus par operation, il ne gagne qu'a partir de n ~ 10^16.3 (Mesure)


entiers n=2^64: ancien ~64.0, nouveau ~63.999999999973376, gain 1.000000000000416x (Mesure)


entiers n=2^1000000: ancien ~1000000.0, nouveau ~999999.999998618033715, gain 1.000000000001382x (Mesure)


-> la victoire asymptotique est exacte mais invisible a toute taille realiste (Mesure)


## 5. 3SUM et SETH — le contrepoint humain de la veille

Le **5 octobre 2026 au soir**, Virginia Williams et Josh Alman ont posté sur arXiv un préprint qui résout **3SUM en $O(n^{1.9992})$** et **All-Pairs Shortest Paths en $O(n^{2.9995})$** — réfutant des conjectures vieux de cinquante ans selon lesquelles les bonnes bornes étaient $n^2$ et $n^3$. Deux leçons dans le billet : (a) **l'idée cruciale a été fournie par un modèle Anthropic**, pas OpenAI — mais Anthropic a choisi une voie de diffusion opposée : plutôt que publier les solutions non digérées au monde, il les a transmises à des mathématiciens humains (Williams et Alman) qui les ont travaillées, vérifiées et signées ; (b) **SETH reste intacte** : le papier « dit explicitement que SETH is unaffected and still open » — 3SUM sous-quadratique ne tue pas la Strong Exponential Time Hypothesis. On mesure ce que $n^2 \to n^{1.9992}$ vaut vraiment, et le scaling réel d'un 3SUM quadratique :


In [5]:
use std::collections::HashMap;

// Generateur MT19937 : reimplementation du flux de CPython (random.Random(seed)).
// `words` compte les mots consommes (diagnostic de conformite du flux).
struct Mt19937 { mt: [u32; 624], idx: usize, words: u64 }

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                      // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624, words: 0 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        self.words += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, DEUX mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 32
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {   // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, stop: u64) -> u64 { self.randbelow(stop) }          // randrange(n)
    fn randrange_de_a(&mut self, debut: i64, fin: i64) -> i64 {                  // randrange(debut, fin) : fin exclu
        debut + self.randbelow((fin - debut) as u64) as i64
    }
    fn randint(&mut self, debut: i64, fin_inclus: i64) -> i64 {                 // randint(debut, fin) : fin inclus
        debut + self.randbelow((fin_inclus - debut + 1) as u64) as i64
    }
}

// Booleens a la Python : "True" / "False".
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Exposant scientifique a la Python : "1e+06" (signe, au moins deux chiffres) --
// le format Rust "{:.0e}" rendrait "1e6".
fn py_exp0(x: f64) -> String {
    let s = format!("{:.0e}", x);
    let mut parties = s.splitn(2, 'e');
    let mantisse = parties.next().unwrap_or("0").to_string();
    let expo = parties.next().unwrap_or("0");
    let signe = if expo.starts_with('-') { "-" } else { "+" };
    let expo_abs: u32 = expo
        .trim_start_matches('+')
        .trim_start_matches('-')
        .parse()
        .unwrap_or(0);
    format!("{}e{}{:02}", mantisse, signe, expo_abs)
}

// (a) L'ecart asymptotique n^2 vs n^1.9992
for n in [1e6f64, 1e12] {
    println!("3SUM n={} : facteur gagne = n^0.0008 = {:.4}x (Mesure)", py_exp0(n), n.powf(0.0008));
}
println!("-> casser la barriere THEORIQUE n^2 ne donne presque rien a taille realiste (Mesure)");

// (b) Le 3SUM quadratique (hash des complements), scaling mesure
fn trois_somme_hash(l: &[i64]) -> bool {
    // O(n^2) : pour chaque paire (i,j), cherche -(L[i]+L[j]) dans la table
    // valeur -> indices, en exigeant un indice k distinct de i et de j --
    // le meme contrat d'indices distincts i < j < k que la triple boucle.
    // Les listes d'indices sont poussees en k croissant et parcourues dans cet
    // ordre : c'est l'ordre de la liste Python, il fait partie du contrat.
    let mut indices: HashMap<i64, Vec<usize>> = HashMap::new();
    for (k, &v) in l.iter().enumerate() {
        indices.entry(v).or_default().push(k);
    }
    let n = l.len();
    for i in 0..n {
        for j in (i + 1)..n {
            let cible = -(l[i] + l[j]);
            if let Some(ks) = indices.get(&cible) {
                for &k in ks.iter() {
                    if k != i && k != j {
                        return true;
                    }
                }
            }
        }
    }
    false
}

// Garde de correction : le hachage doit rendre la meme reponse que la triple
// boucle (indices distincts). Sur [-2, 1, 4], la paire (0,1) vise
// -(L[0]+L[1]) = 1 : la valeur est presente mais son seul indice (1) est deja
// pris par j -> aucun triplet i<j<k ne somme a zero -> False.
// Sur [5, -3, -2, 40] : 5-3-2 = 0 avec trois indices distincts -> True.
assert!(!trois_somme_hash(&[-2, 1, 4]));
assert!(trois_somme_hash(&[5, -3, -2, 40]));
println!("Garde 3SUM : temoins [-2,1,4]->False, [5,-3,-2,40]->True -- indices distincts respectes (Controle)");

// Garde de mesure : entiers TOUS POSITIFS -> aucun triplet nul possible ->
// aucun retour premature, la boucle explore bien les n(n-1)/2 paires (mesure du quadratique complet)
// Un SEUL flux de tirage, comme random.seed(42) appele une fois dans la source :
// les quatre listes se succedent (400, puis 800, 1600, 3200) sans re-seed.
let mut rng = Mt19937::new(42);
let mut temps: Vec<(usize, f64)> = Vec::new();
let mut liste_3200: Vec<i64> = Vec::new();   // gardee pour la comparaison Python / Rust ci-dessous
for n in [400usize, 800, 1600, 3200] {
    let l: Vec<i64> = (0..n).map(|_| rng.randint(1, 1_000_000)).collect();
    let t0 = std::time::Instant::now();
    let r = trois_somme_hash(&l);
    let dt = t0.elapsed().as_secs_f64();
    temps.push((n, dt));
    println!("n={:>5} : {:>8.2} ms, triplet={} (Mesure, parcours complet garanti)", n, dt * 1000.0, py_bool(r));
    if n == 3200 { liste_3200 = l; }
}
let (n1, t1) = temps[1];
let (n2, t2) = temps[3];
let expo_emp = (t2 / t1).ln() / ((n2 as f64) / (n1 as f64)).ln();
println!("Exposant empirique mesure entre n={} et n={} : {:.2} (Mesure - attendu ~2)", n1, n2, expo_emp);

// (c) Ce que Rust apporte : le MEME algorithme sur la MEME liste (n=3200, graine 42,
// quatrieme liste du flux), chronometre sur la MEME machine dans les deux langages.
// Le Python est l'algorithme de la cellule du jumeau, lance ici par `python3` ;
// il refait le flux random.seed(42) et rend sa liste resumee (somme, premier,
// dernier) pour prouver que les deux langages mesurent la meme instance.
// Meilleur de 3 executions de chaque cote (le premier tour chauffe les caches).
let mut meilleur_rust = f64::INFINITY;
for _ in 0..3 {
    let t0 = std::time::Instant::now();
    let r = trois_somme_hash(&liste_3200);
    let dt = t0.elapsed().as_secs_f64();
    assert!(!r);
    if dt < meilleur_rust { meilleur_rust = dt; }
}
let resume_rust = (liste_3200.iter().sum::<i64>(), liste_3200[0], liste_3200[liste_3200.len() - 1]);
const SCRIPT_PY: &str = r#"
import random, time
random.seed(42)
for n in (400, 800, 1600):
    [random.randint(1, 10**6) for _ in range(n)]
L = [random.randint(1, 10**6) for _ in range(3200)]
def trois_somme_hash(L):
    indices = {}
    for k, v in enumerate(L):
        indices.setdefault(v, []).append(k)
    n = len(L)
    for i in range(n):
        for j in range(i + 1, n):
            for k in indices.get(-(L[i] + L[j]), ()):
                if k != i and k != j:
                    return True
    return False
meilleur = float("inf")
for _ in range(3):
    t0 = time.perf_counter()
    assert not trois_somme_hash(L)
    meilleur = min(meilleur, time.perf_counter() - t0)
print(sum(L), L[0], L[-1], meilleur)
"#;
let sortie_py = std::process::Command::new("python3").arg("-c").arg(SCRIPT_PY).output();
match sortie_py {
    Ok(o) if o.status.success() => {
        let texte = String::from_utf8_lossy(&o.stdout).to_string();
        let champs: Vec<&str> = texte.split_whitespace().collect();
        let resume_py = (
            champs[0].parse::<i64>().unwrap_or(-1),
            champs[1].parse::<i64>().unwrap_or(-1),
            champs[2].parse::<i64>().unwrap_or(-1),
        );
        let meilleur_py: f64 = champs[3].parse().unwrap_or(f64::NAN);
        assert_eq!(resume_py, resume_rust, "Python et Rust ne mesurent pas la meme liste");
        println!("Meme instance des deux cotes : n=3200, somme={}, premier={}, dernier={} (Controle)",
                 resume_rust.0, resume_rust.1, resume_rust.2);
        println!("Ce que Rust apporte : 3SUM quadratique, n=3200, meme liste, meme machine -- Python {:.1} ms contre Rust {:.1} ms, Rust {:.1}x plus rapide (Mesure, meilleur de 3)",
                 meilleur_py * 1000.0, meilleur_rust * 1000.0, meilleur_py / meilleur_rust);
        println!("-> le facteur constant gagne par la compilation depasse de loin le n^0.0008 de la barriere cassee (Mesure, comparer a la ligne (a))");
    }
    _ => {
        println!("python3 indisponible ici : comparaison Python / Rust non mesuree sur cette machine.");
        println!("Rust seul : n=3200 en {:.1} ms (Mesure, meilleur de 3)", meilleur_rust * 1000.0);
    }
};

3SUM n=1e+06 : facteur gagne = n^0.0008 = 1.0111x (Mesure)


3SUM n=1e+12 : facteur gagne = n^0.0008 = 1.0224x (Mesure)


-> casser la barriere THEORIQUE n^2 ne donne presque rien a taille realiste (Mesure)


Garde 3SUM : temoins [-2,1,4]->False, [5,-3,-2,40]->True -- indices distincts respectes (Controle)


n=  400 :     4.63 ms, triplet=False (Mesure, parcours complet garanti)


n=  800 :    16.14 ms, triplet=False (Mesure, parcours complet garanti)


... (2 outputs left out)

Exposant empirique mesure entre n=800 et n=3200 : 1.92 (Mesure - attendu ~2)


Meme instance des deux cotes : n=3200, somme=1599256840, premier=130522, dernier=315077 (Controle)


Ce que Rust apporte : 3SUM quadratique, n=3200, meme liste, meme machine -- Python 963.8 ms contre Rust 200.6 ms, Rust 4.8x plus rapide (Mesure, meilleur de 3)


-> le facteur constant gagne par la compilation depasse de loin le n^0.0008 de la barriere cassee (Mesure, comparer a la ligne (a))


## 6. Ce qui manque de la liste — la ligne de crête intacte

Le point le plus important pour un cours de complexité : **ce qui n'est PAS dans les 372**. Le billet : *« P≠NP isn't there, nor even P=BPP or NEXP⊄P/poly, and surely not for lack of trying. »* Les plus grands problèmes ouverts de la théorie de la complexité — P vs NP, P=BPP, NEXP⊄P/poly — n'ont pas bougé. Côté Millennium, le billet relève des **progrès partiels** vers Riemann, Hodge (rétracté pour une erreur de signe) et BSD — « the majority of the remaining Millennium Problems » — mais aucune résolution complète. La vague a cassé des conjectures *circulables* (UGC, L=BPL, des exposants) là où des agents pouvaient explorer des espaces de preuve massifs ; elle n'a rien fait aux questions structurelles qui résistent depuis un demi-siècle. « Plateau ou pas plateau » n'est pas la question : la carte de ce qui est tombé et de ce qui tient est la réponse (Cité — billet, §ce qui manque).


## 7. Épistémologie — preuves aliénées, certificats Lean, deux modèles de diffusion

Trois fils se nouent :

- **Preuves aliénées** : pour plusieurs des 372 résultats, « il semble qu'aucun humain n'ait compris la moindre de ces preuves ». Il existe un **certificat Lean** pour certaines d'entre elles — *pas toutes* : la vérification formelle progresse en parallèle de la compréhension, elle ne la remplace pas.
- **AGMAI** : le groupe consultatif (Gowers, Witten…) a publié une déclaration « très soigneusement phrasée, n'endossant ni ne condamnant » ce qu'OpenAI a fait — et le fil révèle qu'AGMAI avait demandé à OpenAI de détailler ressources, nombre de prompts, problèmes tentés et échoués.
- **Deux modèles de diffusion** : OpenAI publie les solutions brutes au monde entier ; Anthropic remet les idées à des mathématiciens choisis qui les digèrent et signent. Le billet pèse les deux : la première accélère tout le monde mais noie la communauté ; la seconde produit des preuves comprises mais « met une entreprise privée en position de choisir les émissaires de l'IA ». Le débat est ouvert — c'est aussi une question de **politique scientifique**, pas seulement de mathématiques (Cité — billet + commentaires #5, #42, #59).


## Gap Mathlib

À la date de rédaction (14/10/2026), aucun de ces résultats (UGC, L=BPL, parité ∉ QAC$^0$, exposants matmul/entiers, 3SUM sous-quadratique) n'a de formalisation référencée dans ce dépôt — les certificats Lean mentionnés par le billet vivent côté OpenAI, pas ici. *(Renvoi du 28/10/2026 : le volet 3SUM est désormais couvert — voir [03e — Barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb), qui lit la formalisation anthropics/formal-math · 3sum-apsp publiée par les auteurs ; les autres volets de la liste restent sans formalisation référencée.)* Arbitrage rendu : le pli 1 de l'EPIC **#19898** (09/10/2026) classe ces résultats en **non-distillation écrite** — motif : un pli exige un socle de lac à confronter, et aucun lac du dépôt ne porte la théorie correspondante ; la matière est consignée **en veille pour la série Complexity elle-même**, ce carnet en est le premier dépôt. Pour l'indécidabilité et Gödel, l'ancre reste [#15066] (livrée, lake `formal_logic_lean`).


## Exercice 1 — étendre le mini-zoo

La table de la section 3 est une liste d'entrées nommées (un dictionnaire côté Python). Écrire `etend_zoo(zoo, nouvelles)` qui ajoute les nouvelles entrées **sans écraser** les existantes (une clé déjà présente ne se remplace pas — elle signale) et retourne la table étendue.


In [6]:
// Le mini-zoo de la section 3 : une entree = (nom, avant, apres, role).
// Les chaines sont 'static : les entrees du carnet sont des litteraux.
type EntreeZoo = (&'static str, (&'static str, &'static str, &'static str));

fn etend_zoo(zoo: &[EntreeZoo], nouvelles: &[EntreeZoo]) -> Option<Vec<EntreeZoo>> {
    // TODO etudiant : retourner un NOUVEAU vec = zoo + nouvelles,
    // sans jamais modifier une entree existante de zoo
    // Etape 1 : copier zoo
    // Etape 2 : pour chaque (cle, valeur) de nouvelles, ajouter seulement si absente
    // Etape 3 : retourner le resultat
    None
}

// Controle attendu : etend_zoo(zoo, [("P=BPP", ("ouvert", "toujours ouvert", "absent de la vague"))])
// garde les 5 entrees initiales et en ajoute une 6e ; etend_zoo(zoo, [("L = BPL", ("x", "x", "x"))])
// ne change PAS l'entree L = BPL existante.
let _ = ();


## Exercice 2 — la bascule d'un crossover d'exposant

Section 4 : le nouvel algorithme en $n^{2.25}$ ne gagne que si sa constante n'est pas trop mauvaise. Écrire `bascule(e_ancien, e_nouveau, c_ancien, c_nouveau)` : le plus petit $n$ (puissance de 2, énumérée) tel que $c_{nouveau} \cdot n^{e_{nouveau}} \leq c_{ancien} \cdot n^{e_{ancien}}$, ou `None` si l'énumération dépasse $2^{64}$.

Le retour est un `Option<u128>` : $2^{64}$ fait partie du domaine et dépasse `u64::MAX`.


In [7]:
// Le crossover d'un schema a surcout de constante : on enumere les puissances
// de 2 et on compare les deux couts en f64. Le retour est Option<u128> :
// 2^64 = 18446744073709551616 depasse u64::MAX (le jumeau C# prend BigInteger?).
fn bascule(e_ancien: f64, e_nouveau: f64, c_ancien: f64, c_nouveau: f64) -> Option<u128> {
    // TODO etudiant : enumerer n = 1, 2, 4, 8, ... jusqu'a 2**64
    // Etape 1 : boucle sur les puissances de 2 (n = 2**k, k de 0 a 64)
    // Etape 2 : retourner le premier n tel que c_nouveau * n**e_nouveau <= c_ancien * n**e_ancien
    // Etape 3 : None si aucun
    None
}

// Controle attendu : bascule(2.373, 2.25, 1.0, 1.0) == 1 (a constantes egales,
// le meilleur exposant gagne des n=1) ; bascule(2.373, 2.25, 1.0, 100.0) vaut
// 2**55 (le crossover du schema a surcout 100x, n^0.123 = 100 -> n ~ 1.8e16, cf. section 4).
let _ = ();


## Exercice 3 — un 3SUM sous-quadratique ? Non : mesurer le quadratique

La cellule de la section 5 mesure le scaling du 3SUM par hachage. Écrire `trois_somme_naif(L)` (trois boucles imbriquées, $O(n^3)$, même réponse booléenne), puis mesurer les deux implémentations sur $n = 500$ et $n = 1000$ (seed 42) et rapporter le rapport des temps.


In [8]:
// La version naive du 3SUM : trois boucles imbriquees, O(n^3).
// La source Python importe random et time : en Rust, l'aleatoire vient du
// generateur MT19937 de la section 5 (Mt19937::new(42) puis randint(1, 1_000_000)
// sur un flux unique) et le chrono de std::time::Instant.
fn trois_somme_naif(l: &[i64]) -> Option<bool> {
    // TODO etudiant : triple boucle i < j < k, retourner True si L[i]+L[j]+L[k] == 0
    // Etape 1 : trois boucles imbriquees avec i < j < k
    // Etape 2 : test de somme nulle, retour immediat si trouve
    None
}

// Controle attendu : sur L = [5, -3, -2, 40], trois_somme_naif(L) == True (5-3-2=0)
// et sur les memes listes seedees que la section 5, le rapport de temps
// naive/hash entre n=500 et n=1000 doit croitre d'un facteur ~2 (n^3/n^2 = n).
let _ = ();


## Conclusion — la description courte d'un théorème lourd

La vague d'octobre 2026, lue depuis ce dépôt, tient dans le miroir déjà posé par le [07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : des agents ont produit des **descriptions courtes** (des preuves de quelques pages, parfois un certificat Lean) d'**énoncés lourds** (UGC, L=BPL, des barrières d'exposants vieille de cinquante ans) — la face « preuves » du tour des grands nombres du [07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb), et leur pendant côté données — le MDL de la série Compression ([Compression-06](../Compression/Compression-06-MDL-Kolmogorov-Python.ipynb)) ; l'inégalité de Kraft elle-même est dans [Compression-01](../Compression/Compression-01-ShannonFano-Prefixe-Python.ipynb) et son jumeau Lean [Compression-01b — Kraft-McMillan](../Compression/Compression_lean/Compression-01b-Kraft-McMillan-Lean.ipynb) : ce qui est rare n'est pas d'écrire court, c'est de **vérifier** ce qui est écrit court. La ligne de crête — P vs NP — n'a pas bougé ; ce qui a changé, c'est le coût de production d'une preuve circulable, et donc la question épistémique : une preuve que personne ne comprend mais qu'un vérificateur formel valide est-elle une compréhension ? (#19898 a arbitré ces résultats non-distillables sur nos lacs à ce jour — la veille vit dans cette série, en prose mesurée.) Le Zoo ([03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb)) reste la carte où lire la hiérarchie que la vague traverse — ni BPL ni QAC$^0$ n'y figurent.


## À retenir

1. **La vague tient en trois nombres mesurés.** 372 résultats publiés sur environ 4000 problèmes posés : taux de résolution 9.3 %, compute de 1 116 h GPT-Pro pour les seuls problèmes résolus — environ 46 jours d'un seul agent non stop.
2. **La ligne de crête n'a pas bougé.** Le mini-zoo par code rend 4 entrées bougées sur 5 — UGC, L=BPL, parité $\notin$ QAC$^0$, loi d'aire 2D gapée — et P vs NP toujours ouverte : la vague a cassé des conjectures circulables, pas les questions structurelles.
3. **Les victoires d'exposants sont exactes et invisibles.** Le produit matriciel gagne un facteur 30 à $n = 10^{12}$ (écart d'exposant $n^{0.123}$) mais ne bascule qu'à $n \sim 10^{16.3}$ si le nouveau schéma coûte 100× plus par opération ; le 3SUM sous $n^2$ ($n^{1.9992}$) ne gagne que 1.0224× à $n = 10^{12}$ ; la multiplication d'entiers sous la barrière $n \log n$ gagne 1.000000000000416× à $n = 2^{64}$.
4. **Le mur quadratique reste quadratique quand on le mesure.** Le 3SUM par hachage, chronométré de $n = 800$ à $n = 3200$, affiche un exposant empirique propre au rejeu — l'asymptotique nouvelle ne change rien à ce qui tourne à taille réaliste.
5. **La frontière Mesuré/Cité traverse tout le carnet.** Le récit (Moshkovitz « cooked », AGMAI, les deux modèles de diffusion) est **Cité** depuis le billet archivé ; aucune de ces preuves n'est formalisée dans ce dépôt (Gap Mathlib), et la correction publique 8000 → 4000 problèmes fait partie de l'épistémologie.


## Vérifiez votre compréhension

1. Pourquoi le carnet conserve-t-il la mention « une première version disait 8000 problèmes, ~5 % » alors que ces chiffres sont faux ?
2. En quoi le résultat matriciel $n^{9/4+o(1)}$ peut-il être « non constructif », et que mesure la bascule à $n \sim 10^{16.3}$ ?
3. Que signifie « SETH intacte » dans le contrepoint Williams-Alman, alors que 3SUM passe sous $n^2$ ?
4. Quelles entrées du mini-zoo ont bougé, et pourquoi celle qui manque est-elle la plus importante pour un cours de complexité ?

<details><summary>Réponses</summary>

1. Parce que la correction publique fait partie de la leçon : même un témoignage direct se trompe dans l'urgence, les commentateurs l'ont corrigé, et la source officielle (github.com/openai/math) dit environ 4000 problèmes posés. Conserver la trace de l'erreur, c'est montrer l'épistémologie en action.
2. Le billet rapporte (commentaire de Bram Cohen) que le résultat est de type existence, à la Ramsey — sans algorithme qu'on peut exécuter. La bascule à $n \sim 10^{16.3}$ suppose un surcoût constant de 100× par opération pour le nouveau schéma : elle dit à partir de quelle taille astronomique l'avantage d'exposant paierait la constante, donc pourquoi la victoire reste invisible en pratique.
3. Le papier de Williams et Alman dit explicitement que SETH est « unaffected and still open » : casser 3SUM ($O(n^{1.9992})$) et All-Pairs Shortest Paths ($O(n^{2.9995})$) réfute ces conjectures-là, pas la Strong Exponential Time Hypothesis — les barrières ne tombent pas ensemble.
4. UGC (prouvée par la vague), L=BPL, parité $\notin$ QAC$^0$, loi d'aire 2D gapée : quatre entrées sur cinq. La manquante est P vs NP, absente de la liste « pas faute d'avoir essayé » — la question structurelle centrale, qui fixe la limite de ce que l'exploration massive d'espaces de preuve obtient.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Compression-06 — MDL : décrire le modèle autant que les données](../Compression/Compression-06-MDL-Kolmogorov-Python.ipynb) — dernière accrétion de la série : le pendant MDL cité en conclusion, écrire court du côté des données cette fois.
- **Dans la même série** : [Complexity-03c — le Zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — la carte où lire L et la hiérarchie qui l'englobe ; ni BPL ni QAC$^0$ n'y figurent ; [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — l'autre lecture à la source d'une annonce d'octobre 2026, surfaces Lean comprises.
- **Carnet voisin d'une autre série** : [Lean-03b — Formalized Formal Logic](../../SymbolicAI/Lean/Lean-03b-Formalized-Formal-Logic-Lean-Python.ipynb) — ce que « vérifié par une machine » veut dire, mot à mot, sur un lac du dépôt (ancre #15066 citée par le Gap Mathlib).
- **Lectures** : Scott Aaronson, « The Mathocalypse » (Shtetl-Optimized, 7/10/2026, mis à jour le 12/10/2026 — copie archivée au gisement partagé, chemin cité en tête de ce carnet) ; le dépôt github.com/openai/math, la source officielle des résultats.